<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan perhitungan fitur density

In [1]:
# Memilih dataset dari menu
%load_ext autoreload
%autoreload 2

import pandas as pd
import ipywidgets as wg
from IPython.display import display, Markdown
from pci_ml.data import (list_datasets, load_data, clean_data, split_data,
                         checkpoint_options, load_checkpoint)

EXPORT_MODE = False

datasets = list_datasets()
dropdown = wg.Dropdown(
    options=datasets,
    value=datasets[0] if datasets else None,
    description="Pilih file :",
    layout=wg.Layout(width="400px"),
)
df = None

if EXPORT_MODE:
    print(f"File terpilih untuk analisis: {dropdown.value}")
else:
    display(dropdown)

# Helper function that adapts based on EXPORT_MODE
def format_adaptive_df(df_input, is_export_mode):
    if df_input is None:
        return ""      
    df_head = df_input.head()
    if not is_export_mode:
        return df_head.to_markdown(index=False)
    left_count = 5
    right_count = 2
    if df_head.shape[1] > (left_count + right_count):
        left_cols = df_head.iloc[:, :left_count]
        right_cols = df_head.iloc[:, -right_count:]
        middle_col = pd.DataFrame({"…": ["…"] * len(df_head)}, index=df_head.index)
        df_display = pd.concat([left_cols, middle_col, right_cols], axis=1)
    else:
        df_display = df_head
    markdown_table = df_display.to_markdown(index=False)
    return f'<div style="font-size: 11px;">\n\n{markdown_table}\n\n</div>'

handle = display(Markdown(""), display_id=True)

def on_select(change):
    global df
    name = change["new"]
    df = None if name is None else load_data(name)
    
    # Pass the EXPORT_MODE variable into the function
    markdown_table = format_adaptive_df(df, EXPORT_MODE)
    handle.update(Markdown(markdown_table))

dropdown.observe(on_select, names="value")
on_select({"new": dropdown.value})

Dropdown(description='Pilih file :', layout=Layout(width='400px'), options=('pci-ml-26.xlsx',), value='pci-ml-…

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   SEGMENT_AREA |   GATOR_CRACK_L |   GATOR_CRACK_M |   GATOR_CRACK_H |   BLEEDING_L |   BLEEDING_M |   BLEEDING_H |   BLOCK_CRACK_L |   BLOCK_CRACK_M |   BLOCK_CRACK_H |   BUMPS_SAGS_L |   BUMPS_SAGS_M |   BUMPS_SAGS_H |   CORRUGATION_L |   CORRUGATION_M |   CORRUGATION_H |   DEPRESSION_L |   DEPRESSION_M |   DEPRESSION_H |   EDGE_CRACK_L |   EDGE_CRACK_M |   EDGE_CRACK_H |   JOINT_REFL_L |   JOINT_REFL_M |   JOINT_REFL_H |   LANE_SHOULDER_L |   LANE_SHOULDER_M |   LANE_SHOULDER_H |   LT_CRACK_L |   LT_CRACK_M |   LT_CRACK_H |   PATCHING_L |   PATCHING_M |   PATCHING_H |   POLISH_AGG |   POTHOLES_L |   POTHOLES_M |   POTHOLES_H |   RAILROAD_L |   RAILROAD_M |   RAILROAD_H |   RUTTING_L |   RUTTING_M |   RUTTING_H |   SHOVING_L |   SHOVING_M |   SHOVING_H |   SLIPPAGE_CRACK_L |   SLIPPAGE_CRACK_M |   SLIPPAGE_CRACK_H |   SWELL_L |   SWELL_M |   SWELL_H |   RAVELLING_L |   RAVELLING_M |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|---------------:|----------------:|----------------:|----------------:|-------------:|-------------:|-------------:|----------------:|----------------:|----------------:|---------------:|---------------:|---------------:|----------------:|----------------:|----------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|------------------:|------------------:|------------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|------------:|------------:|------------:|------------:|------------:|------------:|-------------------:|-------------------:|-------------------:|----------:|----------:|----------:|--------------:|--------------:|--------------:|--------:|
| Guru Bangkol | R001      | S001         |            450 |            0    |           0     |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |            0   |              0 |              0 |           3.15 |            3   |              0 |              0 |              0 |              0 |               0   |              9    |              12.3 |          0   |         0    |            0 |            0 |          1.1 |         2.86 |            0 |          0.4 |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |        0    |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |            450 |            0    |           1.763 |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |            0.8 |              0 |              0 |           0    |           12.1 |              0 |              0 |              0 |              0 |               0   |              6.08 |               0   |          0   |         3.1  |            0 |            0 |          0   |         9.49 |            0 |          0   |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |        1.96 |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |            450 |            0    |           0     |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |            0   |              0 |              0 |           2.4  |            0   |              0 |              0 |              0 |              0 |               3.1 |              0    |               5.4 |          0   |         3.72 |            0 |            0 |          0   |         0    |            0 |          0   |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |        0    |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |            450 |            0.64 |          10.4   |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |            0   |              0 |              0 |           3    |            0   |              0 |              0 |              0 |              0 |               0   |              0    |               0   |          1.4 |         0    |            0 |            0 |          0   |         1.05 |            0 |          0   |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |        0    |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |            450 |            0    |           0     |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |            0   |              0 |              0 |           2.1  |            0   |              0 |              0 |              0 |              0 |               0   |              0    |               0   |          0   |         5.97 |            0 |            0 |          0   |         0    |            0 |          0   |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |        0    |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 96.1874 |

In [2]:
# Membersihkan data dari duplikat, pencilan dan data tidak valid, serta menghitung fitur density
filename = dropdown.value
df, features = clean_data(load_data(filename))

markdown_table = format_adaptive_df(df, EXPORT_MODE)
display(Markdown(markdown_table))

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   GATOR_CRACK_L |   GATOR_CRACK_M |   GATOR_CRACK_H |   BLEEDING_L |   BLEEDING_M |   BLEEDING_H |   BLOCK_CRACK_L |   BLOCK_CRACK_M |   BLOCK_CRACK_H |   BUMPS_SAGS_L |   BUMPS_SAGS_M |   BUMPS_SAGS_H |   CORRUGATION_L |   CORRUGATION_M |   CORRUGATION_H |   DEPRESSION_L |   DEPRESSION_M |   DEPRESSION_H |   EDGE_CRACK_L |   EDGE_CRACK_M |   EDGE_CRACK_H |   JOINT_REFL_L |   JOINT_REFL_M |   JOINT_REFL_H |   LANE_SHOULDER_L |   LANE_SHOULDER_M |   LANE_SHOULDER_H |   LT_CRACK_L |   LT_CRACK_M |   LT_CRACK_H |   PATCHING_L |   PATCHING_M |   PATCHING_H |   POLISH_AGG |   POTHOLES_L |   POTHOLES_M |   POTHOLES_H |   RAILROAD_L |   RAILROAD_M |   RAILROAD_H |   RUTTING_L |   RUTTING_M |   RUTTING_H |   SHOVING_L |   SHOVING_M |   SHOVING_H |   SLIPPAGE_CRACK_L |   SLIPPAGE_CRACK_M |   SLIPPAGE_CRACK_H |   SWELL_L |   SWELL_M |   SWELL_H |   RAVELLING_L |   RAVELLING_M |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|----------------:|----------------:|----------------:|-------------:|-------------:|-------------:|----------------:|----------------:|----------------:|---------------:|---------------:|---------------:|----------------:|----------------:|----------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|------------------:|------------------:|------------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|------------:|------------:|------------:|------------:|------------:|------------:|-------------------:|-------------------:|-------------------:|----------:|----------:|----------:|--------------:|--------------:|--------------:|--------:|
| Guru Bangkol | R001      | S001         |      0          |      0          |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |     0          |              0 |              0 |     0.007      |     0.00666667 |              0 |              0 |              0 |              0 |        0          |         0.02      |         0.0273333 |   0          |   0          |            0 |            0 |   0.00244444 |   0.00635556 |            0 |  0.000888889 |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |  0          |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |      0          |      0.00391778 |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |     0.00177778 |              0 |              0 |     0          |     0.0268889  |              0 |              0 |              0 |              0 |        0          |         0.0135111 |         0         |   0          |   0.00688889 |            0 |            0 |   0          |   0.0210889  |            0 |  0           |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |  0.00435556 |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |      0          |      0          |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |     0          |              0 |              0 |     0.00533333 |     0          |              0 |              0 |              0 |              0 |        0.00688889 |         0         |         0.012     |   0          |   0.00826667 |            0 |            0 |   0          |   0          |            0 |  0           |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |  0          |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |      0.00142222 |      0.0231111  |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |     0          |              0 |              0 |     0.00666667 |     0          |              0 |              0 |              0 |              0 |        0          |         0         |         0         |   0.00311111 |   0          |            0 |            0 |   0          |   0.00233333 |            0 |  0           |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |  0          |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |      0          |      0          |               0 |            0 |            0 |            0 |               0 |               0 |               0 |              0 |              0 |              0 |               0 |               0 |               0 |     0          |              0 |              0 |     0.00466667 |     0          |              0 |              0 |              0 |              0 |        0          |         0         |         0         |   0          |   0.0132667  |            0 |            0 |   0          |   0          |            0 |  0           |            0 |            0 |            0 |            0 |            0 |           0 |           0 |           0 |           0 |  0          |           0 |                  0 |                  0 |                  0 |         0 |         0 |         0 |             0 |             0 |             0 | 96.1874 |

#### 2. Pemisahan data latih dan data uji

In [3]:
# Menentukan parameter dan kriteria pemisahan data
style = {"description_width": "100px"}
box = wg.Layout(width="180px", margin="0 15px")

n_seed = wg.IntText(value=200, description="Jumlah seed:", style=style, layout=box)
test_size = wg.FloatText(value=0.2, description="Test size:", style=style, layout=box)
share_tol = wg.FloatText(value=0.01, description="Tolerance:", style=style, layout=box)
pci_min = wg.FloatText(value=0, description="PCI minimum:", style=style, layout=box)

if EXPORT_MODE:
    print("Parameter Pemisahan Data:")
    print(f"- Jumlah seed     : {n_seed.value}")
    print(f"- Test size       : {test_size.value}")
    print(f"- Share tolerance : {share_tol.value}")
    print(f"- PCI minimum     : {pci_min.value}")
else:
    display(wg.HBox([n_seed, test_size, share_tol, pci_min],
                    layout=wg.Layout(justify_content="center", flex_flow="row wrap")))

def search_params():
    return {
        "n_seed": n_seed.value,
        "test_size": test_size.value,
        "share_tol": share_tol.value,
        "pci_min": pci_min.value,
    }

In [4]:
from IPython.utils import io

# Memisah data sesuai kriteria
if EXPORT_MODE:
    # Run the function silently (hides tqdm progress bars and any internal prints)
    with io.capture_output() as captured:
        result = split_data(df, features, filename, search_params())
else:
    # Run normally with the progress bar visible
    result = split_data(df, features, filename, search_params())

# Print the summary outside the block so it always shows up
print(result["summary"])

Mencari Seeds:   0%|          | 0/200 [00:00<?, ?it/s]

Seed used: 179 (Gap: 0.7556)
Train: 933 rows (79.27%) | Test: 244 rows (20.73%)
Roads: 27 train | 7 test
Saved to: C:\Users\gearb\Documents\pci-ml\datasets\pci-ml-26\n200_t0.2_s0.01_p0.0


#### 3. Pelatihan model *machine learning*

In [5]:
# Memuat data yang sudah dipisah sesuai kriteria
ckpt_dd = wg.Dropdown(
    options=checkpoint_options(dropdown.value),
    description="Split :",
    layout=wg.Layout(width="700px"),
)

train_df = test_df = meta = None

def load_selected_ckpt(ckpt_name):
    global train_df, test_df, features, meta
    if ckpt_name is not None:
        train_df, test_df, features, meta = load_checkpoint(dropdown.value, ckpt_name)

if EXPORT_MODE:
    selected_ckpt = ckpt_dd.value
    load_selected_ckpt(selected_ckpt)
    display(Markdown(f"**Split terpilih:** `{selected_ckpt}`\n\n**Train:** {train_df.shape} | **Test:** {test_df.shape}"))
else:
    display(ckpt_dd)
    ckpt_handle = display(Markdown(""), display_id=True)

    def on_ckpt(change):
        load_selected_ckpt(change["new"])
        ckpt_handle.update(Markdown(
            f"**Train:** {train_df.shape} | **Test:** {test_df.shape}\n\n"
        ))

    ckpt_dd.observe(on_ckpt, names="value")
    if ckpt_dd.value is not None:
        on_ckpt({"new": ckpt_dd.value})

Loading Checkpoints:   0%|          | 0/1 [00:00<?, ?it/s]

Dropdown(description='Split :', layout=Layout(width='700px'), options=(('2026-11-07 | 03:09:38 | seed 179 | ga…

**Train:** (933, 59) | **Test:** (244, 59)



### ***Source code* modul dan fungsi**

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown

# Path to your src directory
src_dir = Path("src")

# Loop through all Python files and render them as Markdown code blocks
for filepath in src_dir.rglob("*.py"): # rglob searches subdirectories too
    
    # Ignore any files inside Jupyter checkpoint folders
    if ".ipynb_checkpoints" in filepath.parts:
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        code = f.read()
    
    # Creates a nice header for the filename and formats the code
    display(Markdown(f"#### `{filepath.name}`\n```python\n{code}\n```\n---"))